# VLM Metrics (Snapshot-Based)

Run this notebook only while attached to the VLM Docker container (attached mode).

This notebook computes VLM annotation metrics with response normalization:
- yes/no -> 1/0
- number -> integer extracted from text

Metrics:
- binary fields: accuracy, precision, recall, F1
- numeric fields: MAE
- additional: valid response rate (correct format)

Ground-truth strategy:
1. Prefer nuImages object-annotation GT (if object mapping is available).
2. Fallback to reference snapshot GT (recommended: OpenAI snapshot) when direct nuImages mapping is unavailable.

For the current schema, quantitative field evaluation includes:
- numeric: `car_count`, `pedestrian_count`
- binary: `has_car` (derived), `has_pedestrian` (derived), `has_reconstruction_zone`


In [ ]:
# Install dependencies (run once per environment)
!pip install -qU pandas scikit-learn matplotlib seaborn


In [ ]:
from __future__ import annotations

import glob
import json
import re
import tarfile
from collections import defaultdict
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import accuracy_score, f1_score, mean_absolute_error, precision_score, recall_score

# =====================
# CONFIG
# =====================
# Resolve repo root for both launch modes:
# - from repository root
# - from notebooks/ directory
CWD = Path.cwd()
if (CWD / "data/nuimages").exists():
    REPO_ROOT = CWD
elif (CWD.parent / "data/nuimages").exists():
    REPO_ROOT = CWD.parent
else:
    REPO_ROOT = CWD

ROOT = str(REPO_ROOT / "data/nuimages")
META = f"{ROOT}/v1.0-train"
SNAPSHOT_GLOB = str(REPO_ROOT / "avsp-vlm-snapshot-*.tar.gz")
LIMIT_OBJECTS = None  # set int if needed

# GT mode options: "auto", "nuimages", "reference_snapshot"
GROUND_TRUTH_MODE = "auto"
REFERENCE_SNAPSHOT_LABEL = None  # e.g. "openai_snapshot_name"; None -> auto-detect

# Optional manual labels for snapshots
SNAPSHOT_LABELS: Dict[str, str] = {}

# Optional CSV to map storage object_id to nuImages sample token/path.
# Supported columns:
# - object_id + sd_token
# - object_id + filename
OBJECT_ID_MAPPING_CSV = None

# Fields evaluated quantitatively
NUMERIC_FIELDS = ["car_count", "pedestrian_count"]
BINARY_FIELDS = ["has_car", "has_pedestrian", "has_reconstruction_zone"]

# Heuristic GT for reconstruction zone from nuImages object categories
RECONSTRUCTION_GT_CATEGORIES = {
    "vehicle.construction",
    "movable_object.trafficcone",
    "movable_object.barrier",
    "human.pedestrian.construction_worker",
}

FIELD_SCHEMA = {
    "car_count": {"type": "number"},
    "description": {"type": "text"},
    "driving_risk_level": {"type": "category", "allowed": {"low", "medium", "high", "critical", "unknown"}},
    "has_crosswalk": {"type": "yes_no"},
    "has_reconstruction_zone": {"type": "yes_no"},
    "lighting_condition": {"type": "category", "allowed": {"daylight", "dusk", "dawn", "night", "artificial_light", "low_light", "unknown"}},
    "main_risk_factor": {"type": "short_text"},
    "pedestrian_count": {"type": "number"},
    "pedestrian_crossing_road": {"type": "yes_no"},
    "scene_type": {"type": "category", "allowed": {"highway", "urban_street", "residential", "intersection", "parking_lot", "rural_road", "tunnel", "bridge", "roundabout", "unknown"}},
    "short_description": {"type": "short_text"},
    "traffic_light_state": {"type": "category", "allowed": {"red", "yellow", "green", "off", "not_visible", "unclear"}},
    "unexpected_object_on_road": {"type": "yes_no"},
    "weather_condition": {"type": "category", "allowed": {"clear", "cloudy", "rain", "snow", "fog", "wet_road", "unknown"}},
    "road_surface_condition": {"type": "category", "allowed": {"dry", "wet", "snowy", "icy", "muddy", "damaged", "unknown"}},
}

YES_SET = {"yes", "y", "true", "1"}
NO_SET = {"no", "n", "false", "0"}

sns.set_theme(style="whitegrid")


In [ ]:
# =====================
# NORMALIZATION + FORMAT VALIDATION
# =====================
def normalize_yes_no(value) -> Tuple[Optional[int], bool]:
    if value is None:
        return None, False
    s = str(value).strip().lower()

    if s in YES_SET:
        return 1, True
    if s in NO_SET:
        return 0, True

    if re.search(r"\byes\b", s):
        return 1, True
    if re.search(r"\bno\b", s):
        return 0, True

    return None, False


def normalize_number(value) -> Tuple[Optional[int], bool]:
    if value is None:
        return None, False

    s = str(value).strip()
    match = re.search(r"-?\d+", s)
    if not match:
        return None, False

    num = int(match.group())
    if num < 0:
        return None, False
    return num, True


def normalize_category(value, allowed: set) -> Tuple[Optional[str], bool]:
    if value is None:
        return None, False

    s = str(value).strip().lower().replace(" ", "_")
    if s in allowed:
        return s, True
    return None, False


def normalize_short_text(value) -> Tuple[Optional[str], bool]:
    if value is None:
        return None, False

    s = str(value).strip()
    if not s:
        return None, False

    words = [w for w in re.split(r"\s+", s) if w]
    valid = 1 <= len(words) <= 12
    return s, bool(valid)


def normalize_text(value) -> Tuple[Optional[str], bool]:
    if value is None:
        return None, False

    s = str(value).strip()
    if not s:
        return None, False

    sentences = [x.strip() for x in re.split(r"[.!?]+", s) if x.strip()]
    valid = 3 <= len(sentences) <= 5
    return s, bool(valid)


def normalize_by_schema(field: str, value) -> Tuple[object, bool]:
    spec = FIELD_SCHEMA.get(field)
    if spec is None:
        s = "" if value is None else str(value).strip()
        return (s if s else None), bool(s)

    t = spec["type"]

    if t == "yes_no":
        return normalize_yes_no(value)
    if t == "number":
        return normalize_number(value)
    if t == "category":
        return normalize_category(value, spec["allowed"])
    if t == "short_text":
        return normalize_short_text(value)
    if t == "text":
        return normalize_text(value)

    s = "" if value is None else str(value).strip()
    return (s if s else None), bool(s)


In [ ]:
# =====================
# LOAD SNAPSHOTS
# =====================
def _read_tar_member_text(tar: tarfile.TarFile, suffix: str, required: bool = True) -> Optional[str]:
    members = [m for m in tar.getmembers() if m.isfile() and m.name.endswith(suffix)]
    if not members:
        if required:
            raise FileNotFoundError(f"Member with suffix '{suffix}' not found in archive")
        return None

    with tar.extractfile(members[0]) as f:
        return f.read().decode("utf-8")


def _snapshot_label(path: str) -> str:
    if path in SNAPSHOT_LABELS:
        return SNAPSHOT_LABELS[path]
    return Path(path).stem


def _parse_ndjson(text: str) -> List[dict]:
    out = []
    for line in text.splitlines():
        line = line.strip()
        if not line:
            continue
        out.append(json.loads(line))
    return out


def load_snapshot_archive(path: str) -> dict:
    with tarfile.open(path, "r:gz") as tar:
        manifest_text = _read_tar_member_text(tar, "manifest.json", required=True)
        fields_text = _read_tar_member_text(tar, "fields.json", required=False)
        vlm_text = _read_tar_member_text(tar, "vlm.ndjson", required=False)
        objects_text = _read_tar_member_text(tar, "objects.ndjson", required=False)

    manifest = json.loads(manifest_text) if manifest_text else {}

    field_defs = {}
    if fields_text:
        fields = json.loads(fields_text)
        for item in fields:
            fn = str(item.get("field_name", "")).strip()
            rt = str(item.get("response_type", "")).strip().lower()
            if fn:
                field_defs[fn] = rt

    pred_df = pd.DataFrame(columns=["object_id"])
    if vlm_text:
        records = []
        for row in _parse_ndjson(vlm_text):
            object_id = str(row.get("object_id", "")).strip()
            values = row.get("values") or {}
            flat = {"object_id": object_id}
            if isinstance(values, dict):
                flat.update(values)
            records.append(flat)
        pred_df = pd.DataFrame(records)

    objects_df = pd.DataFrame(columns=["object_id", "storage_path", "bucket", "key", "object_file"])
    if objects_text:
        obj_records = []
        for row in _parse_ndjson(objects_text):
            obj_records.append(
                {
                    "object_id": str(row.get("object_id", "")).strip(),
                    "storage_path": str(row.get("storage_path", "")).strip(),
                    "bucket": str(row.get("bucket", "")).strip(),
                    "key": str(row.get("key", "")).strip(),
                    "object_file": str(row.get("object_file", "")).strip(),
                }
            )
        objects_df = pd.DataFrame(obj_records)

    return {
        "path": path,
        "label": _snapshot_label(path),
        "manifest": manifest,
        "field_defs": field_defs,
        "pred_df": pred_df,
        "objects_df": objects_df,
    }


snapshot_paths = sorted(glob.glob(SNAPSHOT_GLOB))
if not snapshot_paths:
    raise FileNotFoundError(f"No snapshots found by pattern: {SNAPSHOT_GLOB}")

snapshots = [load_snapshot_archive(p) for p in snapshot_paths]

print(f"Discovered snapshots: {len(snapshots)}")
for s in snapshots:
    print(
        f"- {s['label']}: "
        f"kind={s['manifest'].get('kind')}, "
        f"vlm_rows={len(s['pred_df'])}, "
        f"objects_rows={len(s['objects_df'])}, "
        f"created_at={s['manifest'].get('created_at')}"
    )


In [ ]:
# =====================
# BUILD GROUND TRUTH (AUTO / NUIMAGES / REFERENCE SNAPSHOT)
# =====================
def build_nuimages_gt_by_token() -> pd.DataFrame:
    object_ann = json.load(open(f"{META}/object_ann.json"))
    category_map = {c["token"]: c["name"] for c in json.load(open(f"{META}/category.json"))}
    sample_data = json.load(open(f"{META}/sample_data.json"))
    sample_to_filename = {s["token"]: s["filename"] for s in sample_data}

    image_to_categories = defaultdict(list)
    for ann in object_ann:
        sd_token = ann["sample_data_token"]
        cat = category_map[ann["category_token"]]
        image_to_categories[sd_token].append(cat)

    rows = []
    for sd_token, cats in image_to_categories.items():
        car_count = sum(1 for c in cats if c == "vehicle.car")
        pedestrian_count = sum(1 for c in cats if c.startswith("human.pedestrian."))
        has_reconstruction_zone = int(any(c in RECONSTRUCTION_GT_CATEGORIES for c in cats))

        rows.append(
            {
                "sd_token": sd_token,
                "filename": sample_to_filename.get(sd_token, ""),
                "car_count": int(car_count),
                "pedestrian_count": int(pedestrian_count),
                "has_car": int(car_count > 0),
                "has_pedestrian": int(pedestrian_count > 0),
                "has_reconstruction_zone": int(has_reconstruction_zone),
            }
        )

    return pd.DataFrame(rows)


def _normalize_candidate_filename(raw: str) -> str:
    s = str(raw or "").strip().lstrip("/")
    if not s:
        return ""
    for marker in ["sweeps/", "samples/"]:
        idx = s.find(marker)
        if idx >= 0:
            return s[idx:]
    return s


def build_object_id_mapping_from_snapshot_objects(snapshots: list) -> pd.DataFrame:
    rows = []
    for snap in snapshots:
        obj = snap["objects_df"]
        if obj.empty:
            continue
        for _, r in obj.iterrows():
            object_id = str(r.get("object_id", "")).strip()
            if not object_id:
                continue
            candidate_sources = [r.get("key", ""), r.get("storage_path", ""), r.get("object_file", "")]
            for src in candidate_sources:
                cand = _normalize_candidate_filename(src)
                if cand:
                    rows.append({"object_id": object_id, "filename": cand})

    if not rows:
        return pd.DataFrame(columns=["object_id", "filename"])

    mapping = pd.DataFrame(rows).drop_duplicates(subset=["object_id", "filename"])
    return mapping


def load_optional_external_mapping() -> pd.DataFrame:
    if not OBJECT_ID_MAPPING_CSV:
        return pd.DataFrame(columns=["object_id", "sd_token", "filename"])

    m = pd.read_csv(OBJECT_ID_MAPPING_CSV)
    cols = {c.lower(): c for c in m.columns}
    if "object_id" not in cols:
        raise ValueError("Mapping CSV must contain 'object_id' column")

    out = pd.DataFrame({"object_id": m[cols["object_id"]].astype(str).str.strip()})

    if "sd_token" in cols:
        out["sd_token"] = m[cols["sd_token"]].astype(str).str.strip()
    if "filename" in cols:
        out["filename"] = m[cols["filename"]].astype(str).str.strip()

    return out


def build_gt_from_nuimages_with_mapping(snapshots: list, nu_gt: pd.DataFrame) -> tuple[pd.DataFrame, str]:
    # 1) Direct overlap: object_id == sd_token
    all_object_ids = set()
    for snap in snapshots:
        if not snap["pred_df"].empty:
            all_object_ids.update(snap["pred_df"]["object_id"].astype(str).tolist())

    direct = nu_gt[nu_gt["sd_token"].isin(all_object_ids)].copy()
    if len(direct) > 0:
        direct = direct.rename(columns={"sd_token": "object_id"})
        return direct[["object_id", "car_count", "pedestrian_count", "has_car", "has_pedestrian", "has_reconstruction_zone"]], "nuImages direct object_id==sd_token"

    # 2) Build mapping object_id -> sd_token via filename
    ext_map = load_optional_external_mapping()
    snap_map = build_object_id_mapping_from_snapshot_objects(snapshots)

    mapping = pd.concat([ext_map, snap_map], ignore_index=True, sort=False)
    if mapping.empty:
        return pd.DataFrame(columns=["object_id", "car_count", "pedestrian_count", "has_car", "has_pedestrian", "has_reconstruction_zone"]), "no mapping available"

    # Fill sd_token from filename if possible
    filename_to_token = dict(zip(nu_gt["filename"], nu_gt["sd_token"]))

    if "filename" in mapping.columns:
        mapping["filename_norm"] = mapping["filename"].map(_normalize_candidate_filename)
    else:
        mapping["filename_norm"] = ""

    if "sd_token" not in mapping.columns:
        mapping["sd_token"] = ""

    mapping["sd_token"] = mapping["sd_token"].astype(str)
    mask_missing = mapping["sd_token"].str.len() == 0
    mapping.loc[mask_missing, "sd_token"] = mapping.loc[mask_missing, "filename_norm"].map(filename_to_token).fillna("")

    mapping = mapping[mapping["sd_token"].str.len() > 0][["object_id", "sd_token"]].drop_duplicates()
    if mapping.empty:
        return pd.DataFrame(columns=["object_id", "car_count", "pedestrian_count", "has_car", "has_pedestrian", "has_reconstruction_zone"]), "mapping exists but no sd_token matches"

    gt = mapping.merge(nu_gt, on="sd_token", how="inner")
    gt = gt[["object_id", "car_count", "pedestrian_count", "has_car", "has_pedestrian", "has_reconstruction_zone"]].drop_duplicates("object_id")
    return gt, "nuImages via object_id mapping"


def select_reference_snapshot(snapshots: list) -> dict:
    if REFERENCE_SNAPSHOT_LABEL:
        for s in snapshots:
            if s["label"] == REFERENCE_SNAPSHOT_LABEL:
                return s
        raise ValueError(f"REFERENCE_SNAPSHOT_LABEL not found: {REFERENCE_SNAPSHOT_LABEL}")

    # Auto-priority: openai/gpt in label
    for s in snapshots:
        lbl = s["label"].lower()
        if "openai" in lbl or "gpt" in lbl:
            return s

    return snapshots[0]


def build_gt_from_reference_snapshot(snapshots: list) -> tuple[pd.DataFrame, str]:
    ref = select_reference_snapshot(snapshots)
    pred_df = ref["pred_df"].copy()

    rows = []
    for _, r in pred_df.iterrows():
        object_id = str(r.get("object_id", "")).strip()
        if not object_id:
            continue

        car_cnt, car_ok = normalize_number(r.get("car_count"))
        ped_cnt, ped_ok = normalize_number(r.get("pedestrian_count"))
        recon, recon_ok = normalize_yes_no(r.get("has_reconstruction_zone"))

        rows.append(
            {
                "object_id": object_id,
                "car_count": car_cnt if car_ok else np.nan,
                "pedestrian_count": ped_cnt if ped_ok else np.nan,
                "has_car": (int(car_cnt > 0) if car_ok else np.nan),
                "has_pedestrian": (int(ped_cnt > 0) if ped_ok else np.nan),
                "has_reconstruction_zone": (int(recon) if recon_ok else np.nan),
            }
        )

    gt = pd.DataFrame(rows).drop_duplicates("object_id")
    return gt, f"reference snapshot: {ref['label']}"


def resolve_ground_truth(snapshots: list) -> tuple[pd.DataFrame, str]:
    nu_gt = build_nuimages_gt_by_token()

    if GROUND_TRUTH_MODE == "nuimages":
        gt, reason = build_gt_from_nuimages_with_mapping(snapshots, nu_gt)
        return gt, reason

    if GROUND_TRUTH_MODE == "reference_snapshot":
        gt, reason = build_gt_from_reference_snapshot(snapshots)
        return gt, reason

    # auto
    gt_nu, reason_nu = build_gt_from_nuimages_with_mapping(snapshots, nu_gt)
    if len(gt_nu) > 0:
        return gt_nu, reason_nu

    gt_ref, reason_ref = build_gt_from_reference_snapshot(snapshots)
    return gt_ref, reason_ref + " (auto fallback)"


gt_df, gt_reason = resolve_ground_truth(snapshots)
if LIMIT_OBJECTS is not None:
    gt_df = gt_df.head(int(LIMIT_OBJECTS)).copy()

print(f"GT mode used: {gt_reason}")
print(f"GT rows: {len(gt_df)}")
print(gt_df.head(3))


In [ ]:
# =====================
# EVALUATION
# =====================
def evaluate_snapshot(snapshot: dict, gt_df: pd.DataFrame):
    model_label = snapshot["label"]
    pred_df = snapshot["pred_df"].copy()

    if pred_df.empty or gt_df.empty:
        empty = pd.DataFrame()
        return empty, empty, empty, empty

    pred_df["object_id"] = pred_df["object_id"].astype(str)
    merged = gt_df.merge(pred_df, on="object_id", how="inner", suffixes=("_gt", "_pred"))

    metric_rows = []

    for _, row in merged.iterrows():
        object_id = row["object_id"]

        # numeric
        car_pred, car_valid = normalize_number(row.get("car_count_pred"))
        ped_pred, ped_valid = normalize_number(row.get("pedestrian_count_pred"))

        car_true = row.get("car_count_gt")
        if pd.notna(car_true):
            metric_rows.append(
                {
                    "model": model_label,
                    "object_id": object_id,
                    "field": "car_count",
                    "task_type": "number",
                    "y_true": int(car_true),
                    "y_pred": car_pred,
                    "valid": bool(car_valid),
                }
            )

        ped_true = row.get("pedestrian_count_gt")
        if pd.notna(ped_true):
            metric_rows.append(
                {
                    "model": model_label,
                    "object_id": object_id,
                    "field": "pedestrian_count",
                    "task_type": "number",
                    "y_true": int(ped_true),
                    "y_pred": ped_pred,
                    "valid": bool(ped_valid),
                }
            )

        # derived binary from counts
        has_car_true = row.get("has_car_gt")
        if pd.notna(has_car_true):
            metric_rows.append(
                {
                    "model": model_label,
                    "object_id": object_id,
                    "field": "has_car",
                    "task_type": "binary",
                    "y_true": int(has_car_true),
                    "y_pred": (int(car_pred > 0) if car_valid else None),
                    "valid": bool(car_valid),
                }
            )

        has_ped_true = row.get("has_pedestrian_gt")
        if pd.notna(has_ped_true):
            metric_rows.append(
                {
                    "model": model_label,
                    "object_id": object_id,
                    "field": "has_pedestrian",
                    "task_type": "binary",
                    "y_true": int(has_ped_true),
                    "y_pred": (int(ped_pred > 0) if ped_valid else None),
                    "valid": bool(ped_valid),
                }
            )

        # direct yes/no field
        recon_true = row.get("has_reconstruction_zone_gt")
        if pd.notna(recon_true):
            recon_pred, recon_valid = normalize_yes_no(row.get("has_reconstruction_zone_pred"))
            metric_rows.append(
                {
                    "model": model_label,
                    "object_id": object_id,
                    "field": "has_reconstruction_zone",
                    "task_type": "binary",
                    "y_true": int(recon_true),
                    "y_pred": recon_pred,
                    "valid": bool(recon_valid),
                }
            )

    if not metric_rows:
        empty = pd.DataFrame()
        return empty, empty, empty, empty

    metrics_long = pd.DataFrame(metric_rows)
    metrics_long["y_pred_filled"] = metrics_long["y_pred"].apply(lambda x: 0 if pd.isna(x) else int(x))

    # format validity for all snapshot fields
    format_rows = []
    all_pred_fields = [c for c in pred_df.columns if c != "object_id"]
    for _, r in pred_df.iterrows():
        for field in all_pred_fields:
            _, ok = normalize_by_schema(field, r.get(field))
            format_rows.append(
                {
                    "model": model_label,
                    "object_id": r["object_id"],
                    "field": field,
                    "valid": bool(ok),
                }
            )

    format_long = pd.DataFrame(format_rows)

    # binary summary
    binary_part = metrics_long[metrics_long["task_type"] == "binary"].copy()
    binary_rows = []
    for (model, field), grp in binary_part.groupby(["model", "field"], sort=False):
        y_true = grp["y_true"].astype(int).to_numpy()
        y_pred = grp["y_pred_filled"].astype(int).to_numpy()
        valid_rate = float(grp["valid"].mean()) if len(grp) > 0 else float("nan")

        binary_rows.append(
            {
                "model": model,
                "field": field,
                "samples": int(len(grp)),
                "accuracy": float(accuracy_score(y_true, y_pred)),
                "precision": float(precision_score(y_true, y_pred, zero_division=0)),
                "recall": float(recall_score(y_true, y_pred, zero_division=0)),
                "f1": float(f1_score(y_true, y_pred, zero_division=0)),
                "valid_rate": valid_rate,
            }
        )
    binary_summary = pd.DataFrame(binary_rows)

    # numeric summary
    numeric_part = metrics_long[metrics_long["task_type"] == "number"].copy()
    numeric_rows = []
    for (model, field), grp in numeric_part.groupby(["model", "field"], sort=False):
        y_true = grp["y_true"].astype(int).to_numpy()
        y_pred = grp["y_pred_filled"].astype(int).to_numpy()
        valid_mask = grp["valid"].astype(bool).to_numpy()
        valid_rate = float(valid_mask.mean()) if len(valid_mask) > 0 else float("nan")

        mae_all = float(mean_absolute_error(y_true, y_pred))
        mae_valid_only = float(mean_absolute_error(y_true[valid_mask], y_pred[valid_mask])) if valid_mask.any() else float("nan")

        numeric_rows.append(
            {
                "model": model,
                "field": field,
                "samples": int(len(grp)),
                "mae": mae_all,
                "mae_valid_only": mae_valid_only,
                "valid_rate": valid_rate,
            }
        )
    numeric_summary = pd.DataFrame(numeric_rows)

    return metrics_long, format_long, binary_summary, numeric_summary


all_metrics_long = []
all_format_long = []
all_binary_summary = []
all_numeric_summary = []

for snapshot in snapshots:
    metrics_long, format_long, binary_summary, numeric_summary = evaluate_snapshot(snapshot, gt_df)
    if not metrics_long.empty:
        all_metrics_long.append(metrics_long)
    if not format_long.empty:
        all_format_long.append(format_long)
    if not binary_summary.empty:
        all_binary_summary.append(binary_summary)
    if not numeric_summary.empty:
        all_numeric_summary.append(numeric_summary)

if not all_metrics_long:
    raise RuntimeError(
        "No evaluable overlap between snapshots and GT. "
        "Provide object_id mapping CSV or use a reference snapshot GT."
    )

metrics_long_df = pd.concat(all_metrics_long, ignore_index=True)
format_long_df = pd.concat(all_format_long, ignore_index=True) if all_format_long else pd.DataFrame(columns=["model", "object_id", "field", "valid"])
binary_summary_df = pd.concat(all_binary_summary, ignore_index=True) if all_binary_summary else pd.DataFrame()
numeric_summary_df = pd.concat(all_numeric_summary, ignore_index=True) if all_numeric_summary else pd.DataFrame()

format_overall = (
    format_long_df.groupby("model", as_index=False)["valid"]
    .mean()
    .rename(columns={"valid": "format_valid_rate_overall"})
)

binary_overall = (
    binary_summary_df.groupby("model", as_index=False)[["accuracy", "precision", "recall", "f1", "valid_rate"]]
    .mean()
    .rename(columns={"valid_rate": "binary_valid_rate"})
)

numeric_overall = (
    numeric_summary_df.groupby("model", as_index=False)[["mae", "mae_valid_only", "valid_rate"]]
    .mean()
    .rename(columns={"valid_rate": "numeric_valid_rate"})
)

overall_summary_df = format_overall.merge(binary_overall, on="model", how="outer").merge(numeric_overall, on="model", how="outer")

print("Evaluation completed.")
print(f"GT mode: {gt_reason}")
print(f"Metrics rows: {len(metrics_long_df)}")
print(f"Unique evaluated objects: {metrics_long_df['object_id'].nunique()}")


In [ ]:
# =====================
# TABLES
# =====================
pd.set_option("display.max_columns", 200)

print("Binary metrics (per field):")
display(binary_summary_df.sort_values(["model", "field"]).reset_index(drop=True))

print("Numeric metrics (per field):")
display(numeric_summary_df.sort_values(["model", "field"]).reset_index(drop=True))

print("Format validity by field:")
format_by_field_df = (
    format_long_df.groupby(["model", "field"], as_index=False)["valid"]
    .mean()
    .rename(columns={"valid": "valid_rate"})
)
display(format_by_field_df.sort_values(["model", "field"]).reset_index(drop=True))

print("Overall model summary:")
display(overall_summary_df.sort_values("model").reset_index(drop=True))


In [ ]:
# =====================
# PLOTS
# =====================
def plot_binary_metrics(binary_df: pd.DataFrame):
    if binary_df.empty:
        print("No binary metrics to plot.")
        return

    metrics = ["accuracy", "precision", "recall", "f1"]
    fig, axes = plt.subplots(2, 2, figsize=(14, 9), sharey=True)
    axes = axes.flatten()

    for ax, metric in zip(axes, metrics):
        sns.barplot(data=binary_df, x="field", y=metric, hue="model", ax=ax)
        ax.set_title(f"Binary {metric.upper()} by field")
        ax.set_xlabel("")
        ax.set_ylim(0, 1)
        ax.tick_params(axis="x", rotation=20)

    handles, labels = axes[0].get_legend_handles_labels()
    for ax in axes:
        if ax.get_legend() is not None:
            ax.get_legend().remove()

    fig.legend(handles, labels, loc="upper center", ncol=max(1, len(labels)))
    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.show()


def plot_numeric_mae(numeric_df: pd.DataFrame):
    if numeric_df.empty:
        print("No numeric metrics to plot.")
        return

    plt.figure(figsize=(10, 5))
    sns.barplot(data=numeric_df, x="field", y="mae", hue="model")
    plt.title("Numeric MAE by field (lower is better)")
    plt.xlabel("")
    plt.ylabel("MAE")
    plt.tight_layout()
    plt.show()


def plot_valid_rate_heatmap(format_df: pd.DataFrame):
    if format_df.empty:
        print("No format-validity data to plot.")
        return

    pivot = (
        format_df.groupby(["model", "field"])["valid"]
        .mean()
        .reset_index()
        .pivot(index="model", columns="field", values="valid")
        .fillna(0.0)
    )

    plt.figure(figsize=(max(10, 0.7 * len(pivot.columns)), max(3, 0.7 * len(pivot.index))))
    sns.heatmap(pivot, annot=True, fmt=".2f", cmap="YlGnBu", vmin=0, vmax=1)
    plt.title("Valid response rate by field")
    plt.xlabel("Field")
    plt.ylabel("Model / Snapshot")
    plt.tight_layout()
    plt.show()


def plot_overall_valid_rate(overall_df: pd.DataFrame):
    if overall_df.empty:
        print("No overall summary to plot.")
        return

    plt.figure(figsize=(8, 4))
    sns.barplot(data=overall_df.sort_values("model"), x="model", y="format_valid_rate_overall")
    plt.ylim(0, 1)
    plt.title("Overall valid response rate")
    plt.ylabel("Valid rate")
    plt.xlabel("")
    plt.xticks(rotation=20)
    plt.tight_layout()
    plt.show()


plot_binary_metrics(binary_summary_df)
plot_numeric_mae(numeric_summary_df)
plot_valid_rate_heatmap(format_long_df)
plot_overall_valid_rate(overall_summary_df)


In [ ]:
# =====================
# OPTIONAL DEBUG: WORST NUMERIC ERRORS
# =====================
num_part = metrics_long_df[metrics_long_df["task_type"] == "number"].copy()
if num_part.empty:
    print("No numeric rows for debug.")
else:
    num_part["y_pred_effective"] = num_part["y_pred"].apply(lambda x: 0 if pd.isna(x) else int(x))
    num_part["abs_error"] = (num_part["y_true"] - num_part["y_pred_effective"]).abs()

    for (model, field), grp in num_part.groupby(["model", "field"], sort=False):
        print()
        print(f"=== {model} | {field} | top-10 absolute errors ===")
        display(grp.sort_values("abs_error", ascending=False).head(10)[["object_id", "y_true", "y_pred", "valid", "abs_error"]])
